---
title: Bipower variation
execute:
  enabled: true
---

`q.indicator.bipower_variation` estimates the continuous component of price variation from adjacent absolute returns, with a finite-sample adjustment. It accepts one finite return window and returns a scalar.

This example applies it to trailing 21-session daily log-return windows for AAPL and SPY over the latest five shared years.

In [1]:
import pandas as pd

import qrt as q

prices = pd.concat(
    {
        "AAPL": q.data.datasets.load("aapl")["close"],
        "SPY": q.data.datasets.load("spy")["close"],
    },
    axis=1,
    join="inner",
).dropna()
end = prices.index.max()
prices = prices.loc[end - pd.DateOffset(years=5) :]
returns = pd.DataFrame({symbol: pd.Series(q.indicator.log_returns(prices[symbol]), index=prices.index[1:]) for symbol in prices})
returns.tail()

,AAPL,SPY
datetime,,
2026-07-17,0.001439,-0.009946
2026-07-20,-0.021657,-0.001616
2026-07-21,0.003515,0.008307
2026-07-22,-0.005661,-0.001163
2026-07-23,-0.013065,-0.012426


## Calculate the indicator

At least two returns are required because the estimator multiplies adjacent absolute returns.

In [2]:
sample = returns["AAPL"].iloc[-21:]
q.indicator.bipower_variation(sample)

0.009962785949001405

## Compare with SPY

Rolling application turns the scalar estimator into a dated series. The 10,000 multiplier makes squared decimal-return values easier to read.

In [3]:
window = 21
comparison = pd.DataFrame(
    {
        symbol: returns[symbol].rolling(window).apply(q.indicator.bipower_variation, raw=True)
        for symbol in returns
    }
).mul(10_000)
comparison.columns = [f"{symbol} bipower variation" for symbol in comparison]
fig = q.plot.col(
    comparison,
    title=f"AAPL and SPY trailing {window}-session bipower variation",
    ylabel="Bipower variation (×10,000)",
)
fig.show(renderer="notebook_connected")